In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset

# AutoModelForCausalLM -> used to load deepseek model

In [21]:
#from huggingface_hub import login
# This will prompt you for your token
#login()

In [22]:
data = [
    {"text": "Patient: I have a headache.\nDoctor: Please take rest and stay hydrated."},
    {"text": "Patient: My stomach is upset.\nDoctor: Try a light diet and monitor your symptoms."},
    {"text": "Patient: I feel dizzy.\nDoctor: Make sure you sit down and drink some water."}
]

In [ ]:
dataset = Dataset.from_list(data)

model_name = "deepseek-ai/deepseek-coder-1.3b-instruct"  
 
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

In [24]:
def tokenize_function(examples):
    tokens = tokenizer(examples["text"],
                       truncation=True,
                       max_length=64,      
                       padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()  
    return tokens

In [25]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [26]:
training_args = TrainingArguments(
    output_dir="./fine_tuned_model",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    max_steps=10,             
    logging_steps=1,
   # no_cuda=True,             
    fp16=False               
)

In [27]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

In [ ]:
trainer.train()

c:\Users\vijay\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
trainer.save_model('./fast_finetuned_model+{}')
tokenizer.save_pretrained('./fast_finetuned')

In [ ]:
def generate_response(prompt, max_length = 64):
    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(**inputs, max_length = max_length)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
prompt= 'Patient: I have a headache.'
print(("chatbot response:",generate_response(prompt) ))